# 04. Strings and JSON

## What you'll learn

- How to manipulate strings with built-in methods (`.strip()`, `.split()`, `.join()`, `.replace()`)
- How to build dynamic prompts with f-strings and template patterns
- How to use regular expressions (`re`) to extract structured data from messy text
- How to serialize and deserialize JSON with the `json` module
- What JSON schemas are and why agents use them to describe tools
- How to combine template strings with data to assemble multi-part prompts

## Prerequisites

- [01. Python Fundamentals](01_python_fundamentals.ipynb) — variables, types, control flow
- [02. Functions and Scope](02_functions_and_scope.ipynb) — defining and calling functions
- [03. Data Structures](03_data_structures.ipynb) — dicts, lists, nested structures

## Why this matters for agents

Agents live in a world of **text**. Every interaction with an LLM is a string going in and a string coming out. Between those two strings, you need to:

1. **Build prompts** — assemble system messages, user queries, tool descriptions, and conversation history into a single string.
2. **Parse responses** — the LLM returns text that may contain JSON tool calls, structured reasoning (Thought/Action/Observation), or embedded data you need to extract.
3. **Validate structure** — ensure that a tool call has the right shape before you execute it.

This notebook gives you the string and JSON skills that every agent notebook in the core track depends on.

> **Further reading:**
> - [Python `re` module docs](https://docs.python.org/3/library/re.html) — full regex reference
> - [Python `json` module docs](https://docs.python.org/3/library/json.html) — serialization/deserialization
> - [JSON Schema — Getting Started](https://json-schema.org/learn/getting-started-step-by-step) — step-by-step intro to JSON schemas

---

## 1. String Methods: The Everyday Toolkit

LLM responses often come back with extra whitespace, unexpected formatting, or delimiters you need to split on. Python's built-in string methods handle all of this without importing anything.

The four methods you'll use most often when working with agent text:

| Method | What it does | Agent use case |
|--------|-------------|----------------|
| `.strip()` | Remove leading/trailing whitespace | Clean up LLM output before parsing |
| `.split()` | Break a string into a list | Separate lines, tokens, or delimited fields |
| `.join()` | Combine a list into a string | Reassemble conversation history |
| `.replace()` | Swap substrings | Sanitize placeholders or remove markers |

In [ ]:
# Simulated LLM response — note the messy whitespace
raw_response = "  \n  The capital of France is Paris.  \n\n"

# .strip() removes leading and trailing whitespace (including newlines)
clean = raw_response.strip()
print(f"Stripped: '{clean}'")

# .split() breaks on whitespace by default
words = clean.split()
print(f"Words: {words}")

# .split() with a specific delimiter
csv_tools = "search,calculator,weather,code_exec"
tool_list = csv_tools.split(",")
print(f"Tools: {tool_list}")

In [ ]:
# .join() is the inverse of .split() — combine a list into a string
tools = ["search", "calculator", "weather"]
tools_string = ", ".join(tools)
print(f"Available tools: {tools_string}")

# Useful for building multi-line prompts from a list of messages
conversation = [
    "User: What's the weather in Tokyo?",
    "Assistant: Let me check that for you.",
    "User: Thanks!",
]
full_conversation = "\n".join(conversation)
print(f"\n{full_conversation}")

In [ ]:
# .replace() swaps every occurrence of a substring
template = "Hello, {USER_NAME}! You have {TOOL_COUNT} tools available."
filled = template.replace("{USER_NAME}", "Alice").replace("{TOOL_COUNT}", "3")
print(filled)

# Handy for cleaning up LLM output that includes unwanted markers
llm_output = "```json\n{\"action\": \"search\"}\n```"
cleaned = llm_output.replace("```json", "").replace("```", "").strip()
print(f"Cleaned JSON: {cleaned}")

A few more string methods that come up frequently:

- `.startswith()` / `.endswith()` — check prefixes and suffixes
- `.lower()` / `.upper()` — normalize case before comparison
- `.find()` — locate a substring (returns -1 if not found, unlike `.index()` which raises an error)

In [ ]:
response = "Action: search\nAction Input: best pizza in NYC"

# Check if the response contains an action
has_action = response.startswith("Action:")
print(f"Has action? {has_action}")

# Split into lines and process each
for line in response.split("\n"):
    key, _, value = line.partition(": ")  # partition splits on first occurrence
    print(f"  {key!r} -> {value!r}")

---

## 2. f-Strings and Prompt Templates

f-strings (formatted string literals) are Python's most readable way to embed expressions inside strings. When building agents, you'll use them constantly to assemble prompts from dynamic data.

The syntax is simple: prefix the string with `f` and put expressions inside `{}`.

In [ ]:
# Basic f-string usage
model_name = "meta-llama/llama-3-8b-instruct"
temperature = 0.7
print(f"Using model: {model_name} with temperature={temperature}")

# f-strings can contain any Python expression
tools = ["search", "calculator", "weather"]
print(f"You have {len(tools)} tools available: {', '.join(tools)}")

# Multi-line f-strings for building prompts
user_query = "What is the population of Tokyo?"
system_prompt = f"""You are a helpful assistant.
You have access to the following tools: {', '.join(tools)}.
Answer the user's question or use a tool if needed.

User: {user_query}
Assistant:"""
print(system_prompt)

### Building a prompt template function

In practice, you'll want reusable prompt templates. A simple function works well for this — no framework required.

In [ ]:
def build_system_prompt(tools, persona="a helpful assistant"):
    """Build a system prompt that describes available tools."""
    tool_descriptions = "\n".join(
        f"  - {tool['name']}: {tool['description']}"
        for tool in tools
    )
    return f"""You are {persona}.

You have access to these tools:
{tool_descriptions}

To use a tool, respond with a JSON object:
{{"tool": "tool_name", "args": {{...}}}}"""


# Define tools as structured data
my_tools = [
    {"name": "search", "description": "Search the web for information"},
    {"name": "calculator", "description": "Evaluate math expressions"},
]

prompt = build_system_prompt(my_tools)
print(prompt)

Notice the `{{` and `}}` in the f-string above. Since `{}` is the f-string interpolation syntax, you need to **double the braces** to produce a literal `{` or `}` in the output. This comes up constantly when your prompts include JSON examples.

---

## 3. Regex Basics: Extracting Structure from Text

Regular expressions (regex) let you search for **patterns** in text, not just exact substrings. When an LLM returns a response that contains a JSON block somewhere inside a larger text output, regex is how you find and extract it.

We use **raw strings** (`r'...'`) for regex patterns so that backslashes are treated literally — `r'\w+'` means "backslash-w-plus" to the regex engine, not an escape sequence. Without the `r` prefix, you'd need to double every backslash (`'\\w+'`).

Python's `re` module provides the regex engine. The three functions you'll use most:

| Function | Returns | Use when |
|----------|---------|----------|
| `re.search(pattern, text)` | First match (or `None`) | You expect one match |
| `re.findall(pattern, text)` | List of all matches | You expect multiple matches |
| `re.sub(pattern, replacement, text)` | Modified string | You want to replace patterns |

In [ ]:
import re

# re.search — find the first occurrence of a pattern
text = "The agent chose action: search with query: 'best pizza'"
match = re.search(r"action: (\w+)", text)

if match:
    print(f"Full match: '{match.group(0)}'")   # the entire match
    print(f"Group 1:    '{match.group(1)}'")    # first capture group (parentheses)

### Key regex patterns for agent work

| Pattern | Meaning | Example match |
|---------|---------|---------------|
| `\w+` | One or more word characters | `search`, `my_tool` |
| `\d+` | One or more digits | `42`, `2024` |
| `.+` | One or more of any character | everything (greedy) |
| `.+?` | One or more of any character (non-greedy) | as little as possible |
| `\{.*?\}` | A `{...}` block (non-greedy) | `{"tool": "search"}` |
| `(...)` | Capture group | extracts what's inside |

In [ ]:
# re.findall — extract all matches
llm_response = """
I found several relevant tools:
- tool: search (for web queries)
- tool: calculator (for math)
- tool: weather (for forecasts)
"""

tool_names = re.findall(r"tool: (\w+)", llm_response)
print(f"Tools mentioned: {tool_names}")

In [ ]:
# Extracting a JSON block from mixed text using re.DOTALL
mixed_response = """
I need to search for this information.

```json
{"tool": "search", "args": {"query": "population of Tokyo"}}
```

Let me call that tool now.
"""

# re.DOTALL makes . match newlines too
json_match = re.search(r"```json\s*(.+?)\s*```", mixed_response, re.DOTALL)
if json_match:
    json_str = json_match.group(1)
    print(f"Extracted JSON:\n{json_str}")

In [ ]:
# re.sub — replace patterns (useful for cleaning LLM output)
messy = "Step  1:   Think   about  the   problem"
clean = re.sub(r"\s+", " ", messy)  # collapse multiple spaces into one
print(f"Cleaned: '{clean}'")

# Remove all markdown bold markers
text_with_bold = "The **agent** should use the **search** tool."
plain = re.sub(r"\*\*(.+?)\*\*", r"\1", text_with_bold)
print(f"Plain: '{plain}'")

---

## 4. The `json` Module: Parsing and Serializing

JSON (JavaScript Object Notation) is the lingua franca of APIs and LLM tool calls. Python's `json` module converts between Python dicts/lists and JSON strings.

Two functions cover 90% of your needs:

| Function | Direction | Input | Output |
|----------|-----------|-------|--------|
| `json.loads(s)` | JSON string -> Python | `'{"a": 1}'` | `{'a': 1}` |
| `json.dumps(obj)` | Python -> JSON string | `{'a': 1}` | `'{"a": 1}'` |

In [ ]:
import json

# json.loads — parse a JSON string into a Python dict
json_string = '{"tool": "search", "args": {"query": "weather in Paris"}}'
tool_call = json.loads(json_string)

print(f"Type: {type(tool_call)}")
print(f"Tool: {tool_call['tool']}")
print(f"Query: {tool_call['args']['query']}")

In [ ]:
# json.dumps — convert a Python object to a JSON string
message = {
    "role": "assistant",
    "content": "Let me search for that.",
    "tool_calls": [
        {"name": "search", "arguments": {"query": "best pizza NYC"}}
    ],
}

# Basic dump
print(json.dumps(message))
print()

# Pretty-printed with indent
print(json.dumps(message, indent=2))

### Handling JSON errors gracefully

LLMs don't always produce valid JSON. When `json.loads()` encounters malformed input, it raises `json.JSONDecodeError`. In agent code, you **must** handle this — a crashing agent is worse than one that retries.

In [ ]:
# Common JSON errors from LLM output
bad_examples = [
    '{"tool": "search", }',           # trailing comma
    "{'tool': 'search'}",              # single quotes (not valid JSON)
    '{tool: "search"}',                # unquoted key
    '{"tool": "search"',               # missing closing brace
]

for bad_json in bad_examples:
    try:
        result = json.loads(bad_json)
    except json.JSONDecodeError as e:
        print(f"FAILED: {bad_json}")
        print(f"  Error: {e.msg} at position {e.pos}")
        print()

In [ ]:
def safe_parse_json(text):
    """Try to parse JSON from text, with common LLM cleanup."""
    # Step 1: strip whitespace
    text = text.strip()

    # Step 2: remove markdown code fences if present
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    # Step 3: try to parse
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e.msg} at position {e.pos}")
        return None


# Test with markdown-wrapped JSON (very common from LLMs)
llm_output = '```json\n{"tool": "calculator", "args": {"expression": "2+2"}}\n```'
parsed = safe_parse_json(llm_output)
print(f"Parsed: {parsed}")

# Test with invalid JSON
bad_output = '{"tool": search}'
parsed = safe_parse_json(bad_output)
print(f"Parsed: {parsed}")

---

## 5. JSON Schemas: Describing the Shape of Data

A **JSON Schema** is a JSON document that describes the expected structure of another JSON document. You don't need a validation library to understand schemas — they're just dicts that say "this field is required, this field is a string, this field is a number."

Why do agents care? Because **tool definitions** are JSON schemas. When you tell an LLM "you have a search tool," you're handing it a schema that describes:
- What arguments the tool accepts
- What types those arguments should be
- Which arguments are required vs optional

In [ ]:
# A tool definition as a JSON schema — this is what gets sent to the LLM
search_tool_schema = {
    "name": "search",
    "description": "Search the web for current information.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query",
            },
            "max_results": {
                "type": "integer",
                "description": "Maximum number of results to return",
                "default": 5,
            },
        },
        "required": ["query"],
    },
}

# Pretty-print it — this is what you'd include in a prompt or API call
print(json.dumps(search_tool_schema, indent=2))

In [ ]:
# Simple manual validation — no library needed
def validate_tool_call(tool_call, schema):
    """Check that a tool call has the required fields with correct types."""
    errors = []
    params = schema["parameters"]
    properties = params["properties"]
    required = params.get("required", [])

    # Check required fields
    for field in required:
        if field not in tool_call.get("args", {}):
            errors.append(f"Missing required field: '{field}'")

    # Check types of provided fields
    type_map = {"string": str, "integer": int, "number": (int, float), "boolean": bool}
    for field, value in tool_call.get("args", {}).items():
        if field in properties:
            expected_type = type_map.get(properties[field]["type"])
            if expected_type and not isinstance(value, expected_type):
                errors.append(f"Field '{field}' should be {properties[field]['type']}, got {type(value).__name__}")

    return errors


# Valid tool call
good_call = {"tool": "search", "args": {"query": "weather in Tokyo", "max_results": 3}}
print(f"Good call errors: {validate_tool_call(good_call, search_tool_schema)}")

# Invalid tool call — missing required field
bad_call = {"tool": "search", "args": {"max_results": 3}}
print(f"Bad call errors: {validate_tool_call(bad_call, search_tool_schema)}")

# Invalid tool call — wrong type
wrong_type_call = {"tool": "search", "args": {"query": 123}}
print(f"Wrong type errors: {validate_tool_call(wrong_type_call, search_tool_schema)}")

Here's a more complete example — multiple tool schemas, just like you'd pass to an LLM in a real agent:

In [ ]:
tool_schemas = [
    {
        "name": "search",
        "description": "Search the web for current information.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query"},
            },
            "required": ["query"],
        },
    },
    {
        "name": "calculator",
        "description": "Evaluate a mathematical expression.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Math expression, e.g. '2 + 3 * 4'"},
            },
            "required": ["expression"],
        },
    },
    {
        "name": "weather",
        "description": "Get current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
                "units": {"type": "string", "description": "'celsius' or 'fahrenheit'"},
            },
            "required": ["city"],
        },
    },
]

# Build a tool-description block for a prompt
for tool in tool_schemas:
    params_str = json.dumps(tool["parameters"], indent=2)
    print(f"### {tool['name']}")
    print(f"{tool['description']}")
    print(f"Parameters: {params_str}")
    print()

---

## 6. Template Strings for Multi-Part Prompts

Real agent prompts are assembled from multiple pieces: a system instruction, tool descriptions, conversation history, and the current user query. Let's build a template system that composes all of these into the final prompt.

There are several approaches. We'll look at two practical ones:
1. **f-strings with helper functions** (simple, good for most cases)
2. **`string.Template`** (built-in, useful when templates come from external files)

In [ ]:
# Approach 1: f-strings with helper functions

def format_tool_descriptions(tools):
    """Format tool schemas into a readable block for a prompt."""
    lines = []
    for tool in tools:
        required = tool["parameters"].get("required", [])
        params = tool["parameters"]["properties"]
        param_parts = []
        for name, info in params.items():
            req_marker = " (required)" if name in required else " (optional)"
            param_parts.append(f"    - {name}: {info['description']}{req_marker}")
        lines.append(f"- {tool['name']}: {tool['description']}")
        lines.extend(param_parts)
    return "\n".join(lines)


def format_messages(messages):
    """Format conversation history."""
    return "\n".join(
        f"{msg['role'].capitalize()}: {msg['content']}"
        for msg in messages
    )


def build_full_prompt(tools, messages, user_query):
    """Assemble a complete agent prompt."""
    tool_block = format_tool_descriptions(tools)
    history_block = format_messages(messages)

    return f"""You are a helpful assistant with access to tools.

## Available Tools
{tool_block}

## Instructions
To use a tool, respond with EXACTLY this format:
Action: <tool_name>
Action Input: <json_arguments>

If you don't need a tool, just respond normally.

## Conversation
{history_block}
User: {user_query}
Assistant:"""


# Build and display a full prompt
history = [
    {"role": "user", "content": "Hi there!"},
    {"role": "assistant", "content": "Hello! How can I help you today?"},
]

prompt = build_full_prompt(tool_schemas, history, "What's 25 * 17?")
print(prompt)

In [ ]:
# Approach 2: string.Template — useful when templates are stored externally
from string import Template

# Template uses $variable_name syntax (no f-string needed)
prompt_template = Template(
    """You are $persona.

Tools: $tools

User: $query
Assistant:"""
)

# .substitute() fills in the variables
result = prompt_template.substitute(
    persona="a research assistant",
    tools="search, calculator",
    query="How far is the Moon from Earth?",
)
print(result)
print()

# .safe_substitute() won't raise an error on missing keys
partial = prompt_template.safe_substitute(persona="an agent")
print("Partial (missing keys left as-is):")
print(partial)

---

## Putting It Together: Parse a Simulated LLM Response

Here's the real scenario: your agent sends a prompt to an LLM, and the LLM responds with some reasoning text **plus** a JSON tool call embedded in the response. Your job is to:

1. Detect whether the response contains a tool call
2. Extract the JSON from the surrounding text
3. Parse and validate it
4. Dispatch to the right tool function

This is exactly what you'll build in [Core 03: Tool Use from Scratch](../core/03_tool_use_from_scratch.ipynb).

In [ ]:
# Simulated LLM response with reasoning + tool call
llm_response = """I need to calculate this math expression for the user.
Let me use the calculator tool.

Action: calculator
Action Input: {"expression": "25 * 17 + 3"}
"""

def parse_agent_response(response):
    """
    Parse an LLM response that may contain a tool call in
    Action/Action Input format.

    Returns:
        dict with 'type' ('tool_call' or 'text') and relevant data.
    """
    # Try to extract Action and Action Input
    action_match = re.search(r"^Action:\s*(.+)$", response, re.MULTILINE)
    input_match = re.search(r"^Action Input:\s*(.+)$", response, re.MULTILINE)

    if action_match and input_match:
        tool_name = action_match.group(1).strip()
        raw_input = input_match.group(1).strip()

        # Try to parse the action input as JSON
        try:
            tool_args = json.loads(raw_input)
        except json.JSONDecodeError as e:
            return {
                "type": "error",
                "message": f"Invalid JSON in Action Input: {e.msg}",
                "raw_input": raw_input,
            }

        # Extract the reasoning text (everything before "Action:")
        reasoning = response[:action_match.start()].strip()

        return {
            "type": "tool_call",
            "reasoning": reasoning,
            "tool": tool_name,
            "args": tool_args,
        }

    # No tool call found — treat as plain text
    return {"type": "text", "content": response.strip()}


# Parse the response
result = parse_agent_response(llm_response)
print(json.dumps(result, indent=2))

In [ ]:
# Now dispatch the tool call

def tool_calculator(expression):
    """Evaluate a math expression (simplified, for demo only)."""
    # In a real agent, you'd sandbox this or use a safe evaluator
    allowed_chars = set("0123456789+-*/(). ")
    if not all(c in allowed_chars for c in expression):
        return {"error": "Expression contains disallowed characters"}
    return {"result": eval(expression)}


def tool_search(query):
    """Simulate a search tool."""
    return {"results": [f"Simulated result for: {query}"]}


# Tool registry — maps tool names to functions
tool_registry = {
    "calculator": tool_calculator,
    "search": tool_search,
}


def dispatch_tool_call(parsed_response):
    """Execute a parsed tool call."""
    if parsed_response["type"] != "tool_call":
        return parsed_response

    tool_name = parsed_response["tool"]
    tool_args = parsed_response["args"]

    if tool_name not in tool_registry:
        return {"error": f"Unknown tool: {tool_name}"}

    tool_fn = tool_registry[tool_name]
    return tool_fn(**tool_args)


# End-to-end: parse -> dispatch -> result
parsed = parse_agent_response(llm_response)
print(f"Reasoning: {parsed['reasoning']}")
print(f"Tool: {parsed['tool']}")
print(f"Args: {parsed['args']}")
print()

tool_result = dispatch_tool_call(parsed)
print(f"Tool result: {tool_result}")

In [ ]:
# Test with a plain text response (no tool call)
plain_response = "The capital of France is Paris. No tool needed for that!"
parsed = parse_agent_response(plain_response)
print(f"Response type: {parsed['type']}")
print(f"Content: {parsed['content']}")
print()

# Test with a different tool
search_response = """I'll search for that information.

Action: search
Action Input: {"query": "population of Tokyo 2024"}
"""
parsed = parse_agent_response(search_response)
tool_result = dispatch_tool_call(parsed)
print(f"Tool: {parsed['tool']}")
print(f"Result: {tool_result}")

---

## Try It Yourself

Work through these exercises to solidify what you've learned. Each one practices a skill you'll use in the core agent notebooks.

### Exercise 1: Extract bulleted list items

Write a function that takes a string containing a bulleted list and returns a Python list of the items (stripped of the bullet character and whitespace).

```python
text = """
Here are the steps:
- Search for relevant information
- Analyze the results
- Formulate a response
- Return the answer to the user
"""
# Expected output: ['Search for relevant information', 'Analyze the results',
#                    'Formulate a response', 'Return the answer to the user']
```

In [ ]:
# Exercise 1: Your code here


### Exercise 2: Pretty-print a message dict as JSON

Write a function `format_message(msg)` that takes a message dict (with `role` and `content` keys, and optional `tool_calls`) and returns a nicely formatted JSON string with 2-space indentation and sorted keys.

```python
msg = {"role": "assistant", "content": "Hello!", "tool_calls": None}
# Should print pretty JSON with sorted keys
```

In [ ]:
# Exercise 2: Your code here


### Exercise 3: Extract URLs with regex

Write a function that finds all URLs in a block of text. A URL starts with `http://` or `https://` and continues until it hits whitespace or end of string.

```python
text = """
Check out https://docs.python.org/3/library/re.html for regex docs.
Also see http://example.com/api?key=123 for the API.
"""
# Expected: ['https://docs.python.org/3/library/re.html', 'http://example.com/api?key=123']
```

In [ ]:
# Exercise 3: Your code here


### Exercise 4: Parse Thought / Action / Action Input format

In the ReAct pattern (which you'll build in [Core 05](../core/05_react_agent.ipynb)), the LLM responds with three labeled fields. Write a function that parses this format and returns a dict.

```python
react_response = """Thought: I need to find the current weather in London to answer the user's question.
Action: weather
Action Input: {"city": "London", "units": "celsius"}"""

# Expected: {
#   'thought': "I need to find the current weather in London to answer the user's question.",
#   'action': 'weather',
#   'action_input': {'city': 'London', 'units': 'celsius'}
# }
```

Hint: Use `re.search` with groups to extract each field, then `json.loads` for the Action Input.

In [ ]:
# Exercise 4: Your code here


---

## Summary

In this notebook you learned:

- **String methods** (`.strip()`, `.split()`, `.join()`, `.replace()`) for cleaning and reshaping text
- **f-strings** for building dynamic prompts with embedded expressions
- **Regular expressions** (`re.search`, `re.findall`, `re.sub`) for pattern-based extraction
- **`json.loads` / `json.dumps`** for converting between JSON strings and Python objects
- **JSON schemas** as a way to describe tool parameters (the foundation of function calling)
- **Template patterns** for assembling multi-part agent prompts from reusable pieces

These are the core text-manipulation skills you'll use in every agent notebook. Next up:

- [05. Error Handling](05_error_handling.ipynb) — `try`/`except`, retry patterns, and basic logging (essential for robust agents)
- [Core 03. Tool Use from Scratch](../core/03_tool_use_from_scratch.ipynb) — where you'll apply all of this to build a real tool-using agent